<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/intermediate/04_model_serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Serving — BentoML & TorchServe

Model serving is about getting predictions to users reliably, quickly, and cost-effectively. Different serving frameworks optimize for different tradeoffs.

**Topics:** BentoML, TorchServe, serving patterns (sync/async/streaming/batch), optimization techniques

In [ ]:
import numpy as np
import time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
import joblib

# Train a model we'll serve
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, 'serving_demo_model.pkl')
print(f'Model accuracy: {model.score(X_test, y_test):.4f}')

## 1. BentoML — ML Serving Framework

In [ ]:
# Install: pip install bentoml

bentoml_code = '''
# service.py — BentoML serving definition
import bentoml
import numpy as np
from bentoml.io import NumpyNdarray, JSON
from pydantic import BaseModel
from typing import List

# Save model to BentoML Model Store
import joblib
model = joblib.load(\'serving_demo_model.pkl\')
bentoml.sklearn.save_model(\'credit_risk_model\', model,
    signatures={\'predict_proba\': {\'batchable\': True, \'batch_dim\': 0}})

# Define service
credit_runner = bentoml.sklearn.get(\'credit_risk_model:latest\').to_runner()

svc = bentoml.Service(\'credit_risk_service\', runners=[credit_runner])

class CreditRequest(BaseModel):
    features: List[List[float]]

class CreditResponse(BaseModel):
    risk_scores: List[float]
    labels: List[str]

@svc.api(input=JSON.from_sample(CreditRequest), output=JSON.from_sample(CreditResponse))
async def predict(input_data: CreditRequest) -> CreditResponse:
    features = np.array(input_data.features)
    
    # Async runner call (non-blocking, supports batching)
    probs = await credit_runner.predict_proba.async_run(features)
    risk_scores = probs[:, 1].tolist()
    labels = [\'HIGH\' if s > 0.5 else \'LOW\' for s in risk_scores]
    
    return CreditResponse(risk_scores=risk_scores, labels=labels)

@svc.api(input=NumpyNdarray(), output=NumpyNdarray())
async def predict_batch(features: np.ndarray) -> np.ndarray:
    """High-throughput batch endpoint."""
    return await credit_runner.predict_proba.async_run(features)
'''

print('=== BentoML Service ===')
print(bentoml_code)

print("""
Commands:
  bentoml serve service:svc --reload     # development
  bentoml build                          # package as Bento
  bentoml containerize credit_risk_service:latest  # Docker image
  bentoml deploy --platform aws-lambda   # deploy to cloud

BentoML adaptive batching:
  Automatically batches concurrent requests up to max_batch_size
  or max_latency — whichever comes first. Dramatically increases
  throughput for GPU models without changing client behavior.
""")

## 2. Serving Patterns & Trade-offs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulate latency distributions for different serving patterns
np.random.seed(42)
n = 1000

# Synchronous: low median latency, high tail
sync_latency = np.random.exponential(scale=20, size=n) + 5

# Batched: higher median, lower tail, higher throughput
batch_latency = np.random.normal(loc=45, scale=8, size=n).clip(25, 100)

# Async queue: variable due to queue depth
async_latency = np.concatenate([
    np.random.normal(30, 5, int(n*0.8)),
    np.random.normal(150, 20, int(n*0.2)),  # queue backup events
])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, data, title, color in zip(axes,
    [sync_latency, batch_latency, async_latency],
    ['Synchronous\n(single prediction)', 'Batched\n(batch_size=32)', 'Async Queue\n(celery workers)'],
    ['steelblue', 'orange', 'green']):
    ax.hist(data, bins=50, color=color, alpha=0.7, edgecolor='white')
    p50 = np.percentile(data, 50)
    p95 = np.percentile(data, 95)
    p99 = np.percentile(data, 99)
    ax.axvline(p50, color='black', ls='-', lw=2, label=f'p50={p50:.0f}ms')
    ax.axvline(p95, color='red', ls='--', lw=2, label=f'p95={p95:.0f}ms')
    ax.axvline(p99, color='darkred', ls=':', lw=2, label=f'p99={p99:.0f}ms')
    ax.set_title(title); ax.set_xlabel('Latency (ms)')
    ax.legend(fontsize=8)

plt.suptitle('Serving Pattern Latency Distributions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('=== Serving Pattern Comparison ===')
print()
patterns = [
    ('Synchronous REST', 'Low p50, high p99', 'Moderate', 'Real-time user-facing', 'Simple CRUD ML APIs'),
    ('Batched', 'Higher p50, lower tail', 'Very High', 'High throughput', 'GPU inference, bulk scoring'),
    ('Async Queue', 'Variable (queue depth)', 'Very High', 'Decouple producer/consumer', 'Long-running training, reports'),
    ('Streaming', 'Lowest TTFT', 'Moderate', 'Real-time partial results', 'LLM token streaming'),
    ('Edge/Client-side', 'Lowest (on-device)', 'High', 'No network, privacy', 'Mobile ML, browser'),
]
print(f'{"Pattern":<20} {"Latency":<25} {"Throughput":<12} {"When to use":<30} {"Examples"}')
print('-' * 115)
for p in patterns:
    print(f'{p[0]:<20} {p[1]:<25} {p[2]:<12} {p[3]:<30} {p[4]}')

## 3. Performance Optimization

In [ ]:
# Key optimization techniques for serving performance

# 1. Prediction caching with Redis
caching_code = '''
import hashlib, json, redis

cache = redis.Redis(host=\'redis\', port=6379, db=0)

def predict_with_cache(features: dict, model, ttl=3600):
    """Cache predictions for identical inputs."""
    cache_key = hashlib.md5(json.dumps(features, sort_keys=True).encode()).hexdigest()
    
    # Check cache
    cached = cache.get(cache_key)
    if cached:
        return json.loads(cached)  # Cache hit!
    
    # Cache miss — compute
    X = preprocess(features)
    result = model.predict_proba(X)[0]
    cache.setex(cache_key, ttl, json.dumps(result.tolist()))
    return result
'''

# 2. Model warming
warming_code = '''
@app.on_event("startup")
async def warm_model():
    """Run dummy prediction to warm JIT/cache."""
    dummy = np.zeros((32, n_features))  # same batch size as production
    for _ in range(5):
        model.predict_proba(dummy)
    logger.info("Model warmed up")
'''

# 3. Benchmark different optimization levels
def benchmark_optimizations():
    batch_sizes = [1, 8, 32, 128]
    n_repeats = 200
    results = {}
    
    for bs in batch_sizes:
        X_bench = np.random.randn(bs, 20)
        times = []
        for _ in range(n_repeats):
            start = time.perf_counter()
            model.predict_proba(X_bench)
            times.append(time.perf_counter() - start)
        
        median_ms = np.median(times) * 1000
        tps = bs / np.median(times)
        results[bs] = {'latency_ms': median_ms, 'throughput': tps}
        print(f'Batch={bs:>5} | Latency={median_ms:>7.2f}ms | Throughput={tps:>8.0f} pred/s')
    
    return results

print('=== Throughput Benchmark ===')
results = benchmark_optimizations()

print()
print('Top ML serving optimization techniques:')
opts = [
    ('Adaptive batching', 'Group concurrent requests → 10-100x GPU throughput'),
    ('ONNX export', 'Framework-agnostic, optimized runtime (2-4x speedup)'),
    ('Model quantization', 'INT8/FP16 → 2-4x speedup, 50-75% memory reduction'),
    ('TensorRT (NVIDIA)', 'GPU-optimized inference → 5-10x speedup'),
    ('Result caching', 'Redis cache for repeated inputs → <1ms response'),
    ('Connection pooling', 'Reuse DB connections → eliminates connection overhead'),
    ('Async I/O', 'Non-blocking I/O → handle 10x more concurrent requests'),
    ('Feature precomputation', 'Pre-compute expensive features, cache in feature store'),
]
for technique, benefit in opts:
    print(f'  {technique:<25} → {benefit}')